## Prompt Evaluation Framework

In [37]:
from dotenv import load_dotenv
from anthropic import Anthropic
from statistics import mean

import re
import ast


load_dotenv()

client = Anthropic()


def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)
    
    
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": "claude-sonnet-4-0",
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
         
    message = client.messages.create(**params)
    
    return message.content[0].text


    

In [38]:
import json

def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate project that generate
        Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that
        requires Python, JSON, or a Regex to complete.
        
        Example output:
        '''json
        [
            {
                "task": "Description of task",
                "format": "json" or "python" or "regex"
                
            },
            ...additional
            
        ]
        '''
        * focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code
        
        Please generate 3 objects.
    """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    
    
    return json.loads(text)
        

In [39]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


/var/folders/76/_j8v2z9n45s8vkphwm93shfw0000gn/T/ipykernel_62166/934062213.py:37: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  message = client.messages.create(**params)


In [41]:
def run_prompt(test_case):
    """Merges nthe prompt and test case input, then returns the results"""
    prompt = f"""
        Please solve following task:
        {test_case["task"]}
        
    * Respond only with Python, JSON, or a plain Regex
    * Do not add any commnets or commentry or explanation
    """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    
    output = chat(messages, stop_sequences=["```"])
    return output

def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.
    
    Original Task:
    <task>
    {test_case["task"]}
    </task>
    
    Solution to Evaluate:
    <solution>
    {output}
    </solution>
    
    Output Format:
    provide your evaluation as a structured JSON object with the following fields, in this specific format
    - "strengths": An array of 1-3 key strengths
    - "weeknesses": An array of 1-3 key weeknesses
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10
    
    Respond with JSON. Keep your response concise and direct.
    example response shape:
    {{
        "strengths": string[],
        "weeknesses": string[],
        "reasoning": string,
        "score": number
    }}
    
    """
    messages = []
    
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)
        

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Model Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_score = grade_syntax(output, test_case)
    
    score = (model_score + syntax_score)/2
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

def run_eval(dataset):
    """ Loads the dataset and calls run_test_case with each case"""
    
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    
    print(f"Average Score:{average_score}")
    
    return results
        
    

In [42]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)
    

/var/folders/76/_j8v2z9n45s8vkphwm93shfw0000gn/T/ipykernel_62166/934062213.py:37: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  message = client.messages.create(**params)


Average Score:8.333333333333334


In [43]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n{\n  \"Version\": \"2012-10-17\",\n  \"Statement\": [\n    {\n      \"Effect\": \"Allow\",\n      \"Action\": [\n        \"s3:GetObject\",\n        \"s3:GetObjectVersion\",\n        \"s3:ListBucket\"\n      ],\n      \"Resource\": [\n        \"arn:aws:s3:::my-company-logs\",\n        \"arn:aws:s3:::my-company-logs/*\"\n      ]\n    }\n  ]\n}\n",
    "test_case": {
      "task": "Create a JSON policy document that grants read-only access to all objects in an S3 bucket named 'my-company-logs'",
      "format": "json"
    },
    "score": 9.0,
    "reasoning": "The policy correctly implements read-only access with proper AWS IAM syntax and resource ARNs. It covers the core requirements with s3:GetObject and s3:ListBucket permissions. The inclusion of s3:GetObjectVersion is not wrong but may be excessive for basic read-only needs. Overall, this is a solid, functional policy that meets the primary requirements."
  },
  {
    "output": "\ndef get_service_from_arn(arn):\